In [ ]:
#celda para los imports y para func de carga de dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from pipeline import create_pipeline
from base_pipeline import create_base_pipeline


def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

In [ ]:
df_completo= load_dataset("futbol_uruguayo.csv")

In [ ]:
#PLACEHOLDER: si fueramos a hacer una separacion del dataset, para tener uno de entrenamiento y otro de test, lo hariamos acá

In [ ]:
#creamos el pipeline base
base_pipeline= create_base_pipeline()

base_df_procesado= base_pipeline.fit_transform(df_completo)

print(
    base_df_procesado[
        [
            #"date",
            "home",
            "away",
            "win_rate",
            "result"
        ]
    ].head(20)
)


In [ ]:
#creamos el pipeline 
pipeline= create_pipeline()

df_procesado= pipeline.fit_transform(df_completo)

print(
    df_procesado[
        [
            #"date",
            "home",
            "away",
            "record",
            "last_matches",
            "goal_difference",
            "local_experience",
            "away_experience",
            "record_enough",
        ]
    ].head(20)
)


    home  away  record  last_matches  goal_difference  local_experience  \
0      1    17       0             0                0                 0   
1      7    33       0             0                0                 0   
2     10    31       0             0                0                 0   
3     26    30       0             0                0                 0   
4     27    22       0             0                0                 0   
5      7    22       0             1                1                 1   
6     17    30       1             1                1                 1   
7     26    10       0             0                1                 1   
8     27    33       1             1                1                 1   
9     31     1       0             0                0                 1   
10     7    10       0             0                1                 1   
11    17    31       1             1                1                 1   
12    26    22       1   

In [ ]:
import sys
import os

# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#Hacemos el arbol de decision en base a lo procesado antes
from decisionTree.clasifier import clasifier as DecisionTreeClassifier

modelo = DecisionTreeClassifier(0.003)

atributos = ["record",
            "last_matches",
            "goal_difference",
            "local_experience",
            "away_experience",
            "record_enough"]

modelo.fit(atributos, df_procesado)
modelo.tree.print_tree()

goal_difference
  [0]
    away_experience
      [0]
        local_experience
          [0]
            1
          [3]
            1
      [1]
        local_experience
          [1]
            1
          [3]
            0
      [3]
        last_matches
          [1]
            1
          [0]
            0
          [2]
            0
      [2]
        1
  [1]
    record
      [0]
        away_experience
          [1]
            last_matches
              [1]
                1
              [0]
                0
          [2]
            last_matches
              [0]
                1
              [1]
                1
              [2]
                0
          [3]
            record_enough
              [1]
                last_matches
                  [1]
                    1
                  [0]
                    1
                  [2]
                    1
              [0]
                2
      [1]
        last_matches
          [1]
            1
          [2]
    